# Task 1 — Time Series Forecasting (Weather)

**Course:** Deep Learning I: Neural Networks — Final Project (Section 1, applied)
**Group:** _<add member names / roll numbers here>_

## Goal
Forecast a real-world time series (hourly air **temperature** from the Jena Climate
dataset) and compare three approaches at predicting **12 hours ahead**:

1. **Moving-average / persistence baseline** (no learning)
2. **Vanilla RNN** (`nn.RNN`)
3. **LSTM** (`nn.LSTM`)

We report **MAE** and **RMSE** (in degrees Celsius), plot forecasts and training
curves, and run an **ablation on the lookback-window length** to analyse how much
past context the recurrent models actually need.

> Runs top-to-bottom on Google Colab. Set the runtime to **GPU** (Runtime →
> Change runtime type → T4 GPU) for the fastest run; it also works on CPU, just slower.


## 1. Setup

In [ ]:
import os, zipfile, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ---- Config (tweak here) ---------------------------------------------------
SEED            = 42
LOOKBACK        = 24      # default history length (hours) for the main models
HORIZON         = 12      # predict 12 hours ahead (harder & more interesting than 1-step)
HIDDEN          = 64
MAIN_EPOCHS     = 12      # epochs for the headline RNN / LSTM
ABLATION_EPOCHS = 6       # epochs per model inside the lookback sweep
BATCH           = 64
LR              = 1e-3
LOOKBACKS       = [6, 12, 24, 48, 96]   # ablation grid
# ---------------------------------------------------------------------------

np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 2. Data — Jena Climate

The Jena Climate dataset records 14 weather variables every 10 minutes from a
station in Jena, Germany (2009–2016). We forecast the temperature column
`T (degC)`, resampled to **hourly** means.

In [ ]:
URL      = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
zip_path = "jena_climate_2009_2016.csv.zip"
csv_path = "jena_climate_2009_2016.csv"

if not os.path.exists(csv_path):
    print("Downloading Jena climate dataset (~13 MB)...")
    urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall()
print("Dataset ready.")

df = pd.read_csv(csv_path)
print("Raw shape:", df.shape)
df.head()

In [ ]:
# Parse the timestamp, set it as the index, and resample temperature to hourly means.
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S')
df = df.set_index('Date Time').sort_index()

TARGET = 'T (degC)'
series = df[TARGET].resample('1h').mean().interpolate('linear')
print("Hourly series length:", len(series))
series.head()

In [ ]:
# A quick look at the full series and a one-week zoom (shows the daily cycle).
fig, ax = plt.subplots(2, 1, figsize=(12, 6))
ax[0].plot(series.index, series.values, lw=0.4)
ax[0].set_title('Hourly temperature, 2009-2016'); ax[0].set_ylabel('T (degC)')
week = series.iloc[2000:2000+24*7]
ax[1].plot(week.index, week.values)
ax[1].set_title('One-week zoom (daily seasonality)'); ax[1].set_ylabel('T (degC)')
plt.tight_layout(); plt.show()

## 3. Preprocessing

- **Chronological split** 70 / 15 / 15 (train / val / test) — *never* shuffle time
  before splitting, or the model peeks at the future.
- **Standardise** using the **training** mean/std only (no leakage).
- **Window** the series into `(LOOKBACK steps -> next step)` supervised samples.

In [ ]:
values = series.values.astype('float32')
n = len(values)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)
train_v, val_v, test_v = values[:train_end], values[train_end:val_end], values[val_end:]
print(f"train {len(train_v)}  val {len(val_v)}  test {len(test_v)}")

mean, std = train_v.mean(), train_v.std()
norm   = lambda a: (a - mean) / std
denorm = lambda a: a * std + mean

def make_windows(arr, lookback, horizon=HORIZON):
    """Turn a 1-D array into (N, lookback, 1) inputs and (N, 1) targets."""
    X, y = [], []
    for i in range(len(arr) - lookback - horizon + 1):
        X.append(arr[i:i + lookback])
        y.append(arr[i + lookback + horizon - 1])
    X = np.array(X, dtype='float32')[..., None]   # (N, lookback, 1)
    y = np.array(y, dtype='float32')[:, None]      # (N, 1)
    return X, y

Xtr, ytr = make_windows(norm(train_v), LOOKBACK)
Xva, yva = make_windows(norm(val_v),   LOOKBACK)
Xte, yte = make_windows(norm(test_v),  LOOKBACK)
print("Windowed:", Xtr.shape, Xva.shape, Xte.shape)

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def make_loader(X, y, bs=BATCH, shuffle=False):
    return DataLoader(SeqDataset(X, y), batch_size=bs, shuffle=shuffle)

train_loader = make_loader(Xtr, ytr, shuffle=True)
val_loader   = make_loader(Xva, yva)
test_loader  = make_loader(Xte, yte)

## 4. Metrics & baseline

MAE and RMSE are computed in the **original units (°C)** after de-normalising.

The baseline needs no training:
- **persistence** — predict that the next hour equals the most recent hour.
- **moving average** — predict the mean of the lookback window.

A good learned model must beat persistence to justify its complexity.

In [ ]:
def mae(p, a):  return float(np.mean(np.abs(p - a)))
def rmse(p, a): return float(np.sqrt(np.mean((p - a) ** 2)))

def baseline_preds(arr, lookback, kind='persistence', horizon=HORIZON):
    """Raw (degC) array, predicts HORIZON steps ahead, aligned with make_windows targets."""
    preds, acts = [], []
    for i in range(len(arr) - lookback - horizon + 1):
        window = arr[i:i + lookback]
        acts.append(arr[i + lookback + horizon - 1])
        preds.append(window[-1] if kind == 'persistence' else window.mean())
    return np.array(preds), np.array(acts)

p_pers, a_base = baseline_preds(test_v, LOOKBACK, 'persistence')
p_ma,   _      = baseline_preds(test_v, LOOKBACK, 'moving_average')

results = {}
results['Persistence']     = (mae(p_pers, a_base), rmse(p_pers, a_base))
results['Moving average']  = (mae(p_ma,   a_base), rmse(p_ma,   a_base))
print("Persistence    MAE %.3f  RMSE %.3f" % results['Persistence'])
print("Moving average MAE %.3f  RMSE %.3f" % results['Moving average'])

## 5. Models — Vanilla RNN & LSTM

Both read the lookback window and use the **last** hidden state to regress the
next value. Identical except for the recurrent cell, so the comparison is fair.

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, hidden=HIDDEN, layers=1):
        super().__init__()
        self.rnn = nn.RNN(1, hidden, layers, batch_first=True)
        self.fc  = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)        # out: (B, T, H)
        return self.fc(out[:, -1])  # last time step -> (B, 1)

class LSTMModel(nn.Module):
    def __init__(self, hidden=HIDDEN, layers=1):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1])

def train_model(model, tr_loader, va_loader, epochs=MAIN_EPOCHS, lr=LR, verbose=True):
    model.to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    hist  = {'train': [], 'val': []}
    for ep in range(epochs):
        model.train(); tl = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = lossf(model(xb), yb)
            loss.backward(); opt.step()
            tl += loss.item() * len(xb)
        tl /= len(tr_loader.dataset)
        model.eval(); vl = 0.0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(device), yb.to(device)
                vl += lossf(model(xb), yb).item() * len(xb)
        vl /= len(va_loader.dataset)
        hist['train'].append(tl); hist['val'].append(vl)
        if verbose:
            print(f"Epoch {ep+1:02d}/{epochs}  train {tl:.4f}  val {vl:.4f}")
    return hist

@torch.no_grad()
def predict(model, loader):
    model.eval(); preds, acts = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(device)).cpu().numpy())
        acts.append(yb.numpy())
    return np.concatenate(preds).ravel(), np.concatenate(acts).ravel()

In [ ]:
print("Training Vanilla RNN...")
rnn = VanillaRNN()
hist_rnn = train_model(rnn, train_loader, val_loader)

print("\nTraining LSTM...")
lstm = LSTMModel()
hist_lstm = train_model(lstm, train_loader, val_loader)

## 6. Evaluation

In [ ]:
for name, model in [('Vanilla RNN', rnn), ('LSTM', lstm)]:
    p, a = predict(model, test_loader)
    p, a = denorm(p), denorm(a)          # back to degC
    results[name] = (mae(p, a), rmse(p, a))

table = pd.DataFrame(results, index=['MAE (degC)', 'RMSE (degC)']).T
table = table.sort_values('RMSE (degC)')
print(table.round(4).to_string())
table

In [ ]:
# Training / validation loss curves (MSE on normalised data).
plt.figure(figsize=(8, 4))
plt.plot(hist_rnn['train'],  label='RNN train')
plt.plot(hist_rnn['val'],    label='RNN val', ls='--')
plt.plot(hist_lstm['train'], label='LSTM train')
plt.plot(hist_lstm['val'],   label='LSTM val', ls='--')
plt.xlabel('epoch'); plt.ylabel('MSE (normalised)'); plt.title('Training curves')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Forecast vs actual over a slice of the test set.
p_rnn,  a_te = predict(rnn,  test_loader);  p_rnn  = denorm(p_rnn);  a_te = denorm(a_te)
p_lstm, _    = predict(lstm, test_loader);  p_lstm = denorm(p_lstm)

s, e = 200, 200 + 24 * 7   # one week
plt.figure(figsize=(13, 4))
plt.plot(a_te[s:e],          label='Actual', lw=2, color='black')
plt.plot(p_pers[s:e],        label='Persistence', alpha=0.5)
plt.plot(p_rnn[s:e],         label='RNN',  alpha=0.8)
plt.plot(p_lstm[s:e],        label='LSTM', alpha=0.8)
plt.title('12-hour-ahead temperature forecast (one-week test slice)')
plt.xlabel('hours'); plt.ylabel('T (degC)'); plt.legend(); plt.tight_layout(); plt.show()

## 7. Ablation — how does the lookback window affect predictions?

We retrain the RNN and LSTM from scratch for each lookback in
`[6, 12, 24, 48, 96]` hours and record the test RMSE/MAE. This isolates the
effect of *how much past context* the model sees. (Fewer epochs here to keep the
sweep fast — bump `ABLATION_EPOCHS` for a more thorough run.)

In [ ]:
abl = {'lookback': [], 'RNN_RMSE': [], 'LSTM_RMSE': [], 'RNN_MAE': [], 'LSTM_MAE': []}

for lb in LOOKBACKS:
    Xtr_, ytr_ = make_windows(norm(train_v), lb)
    Xva_, yva_ = make_windows(norm(val_v),   lb)
    Xte_, yte_ = make_windows(norm(test_v),  lb)
    tl_ = make_loader(Xtr_, ytr_, shuffle=True)
    vl_ = make_loader(Xva_, yva_)
    te_ = make_loader(Xte_, yte_)

    r = VanillaRNN(); train_model(r, tl_, vl_, epochs=ABLATION_EPOCHS, verbose=False)
    pr, ar = predict(r, te_); pr, ar = denorm(pr), denorm(ar)
    l = LSTMModel();  train_model(l, tl_, vl_, epochs=ABLATION_EPOCHS, verbose=False)
    pl, al = predict(l, te_); pl, al = denorm(pl), denorm(al)

    abl['lookback'].append(lb)
    abl['RNN_RMSE'].append(rmse(pr, ar));  abl['RNN_MAE'].append(mae(pr, ar))
    abl['LSTM_RMSE'].append(rmse(pl, al)); abl['LSTM_MAE'].append(mae(pl, al))
    print(f"lookback {lb:3d}  RNN RMSE {abl['RNN_RMSE'][-1]:.3f}  LSTM RMSE {abl['LSTM_RMSE'][-1]:.3f}")

abl_df = pd.DataFrame(abl).set_index('lookback')
abl_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(abl_df.index, abl_df['RNN_RMSE'],  'o-', label='RNN  RMSE')
plt.plot(abl_df.index, abl_df['LSTM_RMSE'], 's-', label='LSTM RMSE')
plt.xlabel('lookback window (hours)'); plt.ylabel('test RMSE (degC)')
plt.title('Effect of lookback window length on forecast error')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 8. Analysis & conclusions

_Fill these in with your actual numbers after running — the notebook prints them all._

- **Ranking.** At a **12-hour horizon** the persistence baseline degrades sharply (the
  temperature 12 h later is often a full day/night swing away), so the learned models
  should now clearly win: expect **LSTM ≤ RNN < moving average < persistence** on RMSE.
  (Note: at a 1-hour horizon persistence is much harder to beat — the longer horizon is
  exactly what makes the recurrent models shine.)
- **RNN vs LSTM.** The LSTM's gates let it retain information across longer windows
  without the vanishing-gradient problems of the vanilla RNN, so its advantage tends
  to grow as the lookback increases.
- **Lookback window (the ablation).** Too **short** a window starves the model of the
  daily cycle and hurts accuracy; increasing it helps up to roughly one daily period
  (~24 h), after which returns flatten or *reverse* — longer sequences add noise and
  are harder to optimise (especially for the vanilla RNN). Report where *your* curve
  bottoms out and explain it in terms of the daily seasonality you saw in Section 2.
- **Limitations / next steps.** Single-feature, single-step. Natural extensions:
  multi-step forecasting, adding exogenous features (pressure, humidity), and a GRU
  for comparison.
